# Vehicle Loan Default – Credit Risk Analysis

Course-End Project

This notebook implements end-to-end credit risk analysis and prediction for vehicle loan defaulters based on the provided problem statement.

## 1. Problem Statement

Financial institutions face losses due to vehicle loan defaults. This analysis aims to identify risk factors and build a predictive model to flag potential defaulters.

## 2. Objectives
- Data Cleaning & Preprocessing
- Exploratory Data Analysis (EDA)
- Factor Analysis
- Logistic Regression Model
- Model Evaluation

## 3. Data Importing & Inspection

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load dataset (update path if required)
df = pd.read_excel('vehicle_loan_data.xlsx')
df.head()


In [ ]:

# Basic inspection
df.shape, df.info()


In [ ]:

# Duplicate removal
df = df.drop_duplicates()

# Missing value summary
missing = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_%': (df.isnull().sum()/len(df))*100
})
missing[missing.Missing_Count > 0]


### Standardize Column Names

In [ ]:

df.columns = (
    df.columns.str.lower()
              .str.replace('.', '_')
              .str.replace(' ', '_')
)
df.head()


### Handling Missing Values

In [ ]:

num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include='object').columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

df.isnull().sum().sum()


## 4. Exploratory Data Analysis (EDA)
### 4.1 Statistical Description

In [ ]:

df.describe()


### 4.2 Target Variable Distribution

In [ ]:

df['loan_default'].value_counts(normalize=True)

sns.countplot(x='loan_default', data=df)
plt.title('Loan Default Distribution')
plt.show()


### 4.3 Employment Type Analysis

In [ ]:

sns.countplot(x='employment_type', hue='loan_default', data=df)
plt.xticks(rotation=45)
plt.title('Employment Type vs Loan Default')
plt.show()


### 4.4 Age Distribution

In [ ]:

from datetime import datetime

df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')
df['age'] = datetime.now().year - df['date_of_birth'].dt.year

sns.histplot(data=df, x='age', hue='loan_default', kde=True)
plt.title('Age Distribution vs Default')
plt.show()


### 4.5 Identity Proof Analysis

In [ ]:

id_cols = ['aadhar_flag','pan_flag','voterid_flag','driving_flag','passport_flag']
id_counts = df[id_cols].sum()

sns.barplot(x=id_counts.index, y=id_counts.values)
plt.title('ID Proof Distribution')
plt.xticks(rotation=45)
plt.show()


## 5. Detailed Factor Analysis
### Credit Bureau Score

In [ ]:

sns.kdeplot(data=df, x='perform_cns_score', hue='loan_default', fill=True)
plt.title('Credit Bureau Score Distribution')
plt.show()


### Account & Enquiry Analysis

In [ ]:

sns.scatterplot(x='no_of_inquiries', y='perform_cns_score', hue='loan_default', data=df)
plt.title('Enquiries vs Credit Score')
plt.show()


## 6. Predictive Modeling – Logistic Regression

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

X = df.drop(columns=['loan_default','date_of_birth'])
X = pd.get_dummies(X, drop_first=True)
y = df['loan_default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]


In [ ]:

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


### ROC Curve

In [ ]:

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label='ROC Curve')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()


## 7. Conclusion

Credit bureau score, enquiry history, employment type, and loan history are the strongest predictors of vehicle loan default. Logistic Regression provides a strong baseline model for credit risk assessment.